# Lesson 23: Hyperparameter Tuning & Model Selection

*ML Academy — companion notebook*

Validation curves, grid and random search, Bayesian optimisation, successive halving, nested CV and choosing the simplest good model.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-23.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Seeing one hyperparameter at a time: validation curves

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import validation_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_breast_cancer(return_X_y=True)
model = make_pipeline(StandardScaler(), SVC())
gammas = [0.0001, 0.001, 0.01, 0.1, 1]
train, val = validation_curve(model, X, y, param_name="svc__gamma", param_range=gammas, cv=5)
for g, a, b in zip(gammas, train.mean(axis=1), val.mean(axis=1)):
    print(f"gamma {g:<6}: train accuracy {a:.3f}   cross-validated accuracy {b:.3f}")

gamma 0.0001: train accuracy 0.794   cross-validated accuracy 0.796
gamma 0.001 : train accuracy 0.951   cross-validated accuracy 0.947
gamma 0.01  : train accuracy 0.978   cross-validated accuracy 0.968
gamma 0.1   : train accuracy 0.990   cross-validated accuracy 0.960
gamma 1     : train accuracy 1.000   cross-validated accuracy 0.631


## 3. Grid search

In [2]:
from sklearn.model_selection import GridSearchCV, train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.0001, 0.001, 0.01, 0.1]}
search = GridSearchCV(model, grid, cv=5).fit(X_tr, y_tr)
print("settings tried:", len(search.cv_results_["params"]))
print("best settings:", search.best_params_)
print("best cross-validated accuracy:", round(search.best_score_, 3))
print("test accuracy of the refitted best model:", round(search.score(X_te, y_te), 3))

settings tried: 16
best settings: {'svc__C': 100, 'svc__gamma': 0.001}
best cross-validated accuracy: 0.988
test accuracy of the refitted best model: 0.958


## 4. Random search

In [3]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

dists = {"svc__C": loguniform(0.01, 1000), "svc__gamma": loguniform(1e-5, 1)}
rs = RandomizedSearchCV(model, dists, n_iter=16, cv=5, random_state=0).fit(X_tr, y_tr)
print("best settings:", {k: round(float(v), 4) for k, v in rs.best_params_.items()})
print("best cross-validated accuracy:", round(rs.best_score_, 3), "  test accuracy:", round(rs.score(X_te, y_te), 3))

best settings: {'svc__C': 1.313, 'svc__gamma': 0.017}
best cross-validated accuracy: 0.981   test accuracy: 0.951


## 6. Smarter searches: Bayesian optimisation and successive halving

In [4]:
from sklearn.experimental import enable_halving_search_cv  # noqa: F401  (switches the feature on)
from sklearn.model_selection import HalvingRandomSearchCV

hs = HalvingRandomSearchCV(model, dists, n_candidates=48, factor=3, cv=5, random_state=0).fit(X_tr, y_tr)
for it, (n_cand, n_res) in enumerate(zip(hs.n_candidates_, hs.n_resources_)):
    print(f"round {it + 1}: {n_cand:2d} candidates, each trained on {n_res} rows")
print("best cross-validated accuracy:", round(hs.best_score_, 3))

round 1: 48 candidates, each trained on 20 rows
round 2: 16 candidates, each trained on 60 rows
round 3:  6 candidates, each trained on 180 rows
best cross-validated accuracy: 0.978


## 7. Reporting honestly: nested cross-validation

In [5]:
from sklearn.model_selection import cross_val_score, KFold

inner, outer = KFold(5, shuffle=True, random_state=1), KFold(5, shuffle=True, random_state=2)
tuned = GridSearchCV(model, grid, cv=inner)
nested = cross_val_score(tuned, X, y, cv=outer)
print("nested CV accuracy:", round(nested.mean(), 3), "+/-", round(nested.std(), 3))

nested CV accuracy: 0.979 +/- 0.007


## 8. Choosing between models: the simplest good one

In [6]:
from sklearn.tree import DecisionTreeClassifier

# The one-standard-error rule: prefer the simplest model within 1 SE of the best
depths = list(range(1, 11))
res = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": depths}, cv=10).fit(X_tr, y_tr).cv_results_
mean, se = res["mean_test_score"], res["std_test_score"] / np.sqrt(10)
best = int(np.argmax(mean))
simplest = next(i for i in range(len(depths)) if mean[i] >= mean[best] - se[best])
print("best depth by score:", depths[best], " CV accuracy", round(mean[best], 3), " SE", round(se[best], 3))
print("simplest depth within 1 SE:", depths[simplest], " CV accuracy", round(mean[simplest], 3))

best depth by score: 5  CV accuracy 0.941  SE 0.012
simplest depth within 1 SE: 4  CV accuracy 0.929


## 9. In Python: a tuning recipe

In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import loguniform

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
search = RandomizedSearchCV(pipe, {"logisticregression__C": loguniform(1e-3, 1e3)},
                            n_iter=30, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                            scoring="roc_auc", random_state=0).fit(X_tr, y_tr)
print("best C:", round(float(search.best_params_["logisticregression__C"]), 4))
print("CV ROC-AUC:", round(search.best_score_, 4), "  test ROC-AUC:", round(search.score(X_te, y_te), 4))

best C: 0.3483
CV ROC-AUC: 0.9954   test ROC-AUC: 0.9943


## 12. Going deeper: Bayesian optimisation from scratch

In [8]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")          # the GP optimiser warns about its bounds on tiny data
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

# A made-up "validation score vs log10(learning rate)" curve with its peak near x = -2.3
f = lambda x: np.exp(-(x + 2.3) ** 2 / 0.3) * 0.3 + 0.65 + 0.02 * np.sin(5 * x)
grid = np.linspace(-5, 0, 501)[:, None]

def bayes_opt(n_iter, seed):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-5, 0, size=(3, 1)); y = f(X[:, 0])               # 3 random starting points
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.5, length_scale_bounds=(0.05, 5), nu=2.5)
    gp = GaussianProcessRegressor(kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=3, random_state=0)
    for _ in range(n_iter - 3):
        gp.fit(X, y)
        mu, sd = gp.predict(grid, return_std=True)
        z = (mu - y.max()) / np.maximum(sd, 1e-9)
        ei = (mu - y.max()) * norm.cdf(z) + sd * norm.pdf(z)             # expected improvement
        x_next = grid[np.argmax(ei)]
        X = np.vstack([X, x_next]); y = np.append(y, f(x_next[0]))
    return y.max()

def random_search(n_iter, seed):
    return f(np.random.default_rng(seed).uniform(-5, 0, n_iter)).max()

best = f(grid[:, 0]).max()
for n in [5, 10, 15]:
    bo = np.mean([best - bayes_opt(n, s) for s in range(20)])
    rs = np.mean([best - random_search(n, s) for s in range(20)])
    print(f"{n:2d} trials: average gap to the best score, Bayesian {bo:.4f}   random {rs:.4f}")

 5 trials: average gap to the best score, Bayesian 0.1297   random 0.1190


10 trials: average gap to the best score, Bayesian 0.0103   random 0.0510


15 trials: average gap to the best score, Bayesian 0.0000   random 0.0261


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [9]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Count the grid

Compute how many model fits a 5-fold grid search over this grid performs (not counting the refit).

In [10]:
grid = {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, 1, 10], "kernel": ["rbf", "sigmoid"]}
folds = 5
fits = len(grid) * folds  # ← replace: count every combination
check(fits, 120, hint='Multiply the number of values of each hyperparameter, then by the number of folds.')

❌ Not yet. Hint: Multiply the number of values of each hyperparameter, then by the number of folds.


### Challenge 2: Log-scale samples

Draw 5 learning rates spread evenly on a log scale from 0.001 to 1.

In [11]:
import numpy as np
rates = np.linspace(0.001, 1, 5)  # ← replace with a log-spaced version
result = [round(float(v), 4) for v in rates]
check(result, [0.001, 0.0056, 0.0316, 0.1778, 1.0], hint='np.logspace(start, stop, num) uses powers of ten: 10**-3 to 10**0.')

❌ Not yet. Hint: np.logspace(start, stop, num) uses powers of ten: 10**-3 to 10**0.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Count the grid** — the missing line:

```python
fits = len(grid["C"]) * len(grid["gamma"]) * len(grid["kernel"]) * folds
```

**2. Log-scale samples** — the missing line:

```python
rates = np.logspace(-3, 0, 5)
```